# LLM-Based Orchestration + MCP + Two Databricks Genie Agents

**Goal:** Demonstrate an LLM orchestrator that decides whether to call Genie Space A, Genie Space B, both, or
neither, using their existing managed MCP endpoints.

**No Databricks App, custom MCP server, model or endpoint is created or deployed.** This notebook acts as an MCP
client and an LLM orchestrator. It connects to the two existing Genie MCP endpoints and calls them as tools.

## Architecture

```text
 User question
      |
      v
 Orchestrator LLM (router) ---- JSON: route A | B | BOTH | NONE, one sub-question per space
      |
      +--> MCP tools/call query_space_<A> --> poll_response_<A> until done --> Genie Space A (SQL + rows)
      +--> MCP tools/call query_space_<B> --> poll_response_<B> until done --> Genie Space B (SQL + rows)
      |
      v
 Orchestrator LLM (synthesis) --> final answer, attributed to each space
```

### Prerequisites
- Two existing Genie Spaces.
- Managed Genie MCP endpoints enabled in your workspace (`/api/2.0/mcp/genie/<space_id>`).
- Permission to use both spaces (*CAN RUN* or higher) and their underlying data and SQL warehouse.
- An existing Model Serving chat endpoint for the orchestrator LLM. A Foundation Model API pay-per-token endpoint
  such as `databricks-claude-sonnet-4-5` works; leave the name empty and the notebook picks one.
- Workspace authentication configured according to your organization's policy (in a Databricks notebook this is
  automatic).

### How to run
1. Fill in the two Genie Space IDs in section 2 (the ID is in the space's URL: `.../genie/rooms/<space_id>`;
   pasting the whole URL also works).
2. **Run all.** Section 1 installs the client libraries and restarts Python; then everything runs top to bottom.

## 1. Install dependencies

The MCP Python SDK and the OpenAI client are installed into this notebook's environment only (no cluster or
workspace change). Python then restarts so the new versions are used.

In [ ]:
%pip install --quiet "mcp>=1.9,<2" databricks-sdk openai

In [ ]:
try:
    dbutils.library.restartPython()  # noqa: F821  (Databricks: load the packages just installed)
except NameError:
    pass                             # not on Databricks: nothing to restart

## 2. Configure the two Genie Spaces and the orchestrator model endpoint

| Setting | What to enter |
|---|---|
| `GENIE_A_ID`, `GENIE_B_ID` | **required**: the two Genie Space IDs (or their URLs) |
| `GENIE_A_NAME`, `GENIE_B_NAME` | optional: display names. Empty = the space's title in Genie |
| `GENIE_A_DESCRIPTION`, `GENIE_B_DESCRIPTION` | optional: what data each space holds. **The router decides on this.** Empty = the space's description in Genie |
| `ORCHESTRATOR_ENDPOINT` | optional: an existing chat endpoint. Empty = pick a Foundation Model endpoint (Claude first) |
| `DEMO_QUESTIONS` | optional: your own demo questions. Empty = generated from the spaces' descriptions |

Do not put personal access tokens or secrets directly in this notebook.

In [ ]:
GENIE_A_ID = "<GENIE_SPACE_A_ID>"
GENIE_B_ID = "<GENIE_SPACE_B_ID>"

GENIE_A_NAME = ""            # e.g. "Sales"
GENIE_B_NAME = ""            # e.g. "Inventory"

GENIE_A_DESCRIPTION = ""     # e.g. "Net sales, margin and basket size by store, region and week"
GENIE_B_DESCRIPTION = ""     # e.g. "Stock on hand, out-of-stock rate and days of cover by store and product"

# Existing Databricks Model Serving endpoint for the orchestration LLM (OpenAI Chat Completions compatible).
ORCHESTRATOR_ENDPOINT = ""   # e.g. "databricks-claude-sonnet-4-5"; empty = pick one automatically

# Optional: (question, expected route) pairs; expected is "A", "B", "BOTH" or "NONE".
DEMO_QUESTIONS = [
    # ("What were the main sales trends last quarter?", "A"),
    # ("Which products are most often out of stock?", "B"),
    # ("Did the stores with the highest sales also have the most stock-outs?", "BOTH"),
]

MCP_TIMEOUT_SECONDS = 300    # a Genie tool call can hold the request open while Genie works
GENIE_TIMEOUT_SECONDS = 600  # give up on one Genie answer after this long
POLL_EVERY_SECONDS = 3

## 3. Connect to the workspace and prepare authentication

The Databricks SDK uses the notebook's configured authentication. We reuse the current authorization headers for
the managed Genie MCP calls. For the LLM, the OpenAI client targets the Databricks serving endpoints with the same
workspace host and a token from the SDK's authentication provider.

If your organization requires a different approved authentication method, adapt this cell to match that policy.

In [ ]:
import asyncio
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from contextlib import asynccontextmanager

import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from openai import OpenAI

try:                                             # mcp 2.x
    import httpx2
    from mcp import Client
    from mcp.client.streamable_http import streamable_http_client
    MCP_V2 = True
except ImportError:                              # mcp 1.x, as installed in section 1
    from mcp import ClientSession
    from mcp.client.streamable_http import streamablehttp_client
    MCP_V2 = False

w = WorkspaceClient()
WORKSPACE_HOST = w.config.host.rstrip("/")

PLACEHOLDERS = {"", "<GENIE_SPACE_A_ID>", "<GENIE_SPACE_B_ID>", "<EXISTING_MODEL_SERVING_ENDPOINT>"}


def current_auth_headers():
    return {str(k): str(v) for k, v in w.config.authenticate().items()}


def current_bearer_token():
    headers = current_auth_headers()
    for key, value in headers.items():
        if key.lower() == "authorization" and value.lower().startswith("bearer "):
            return value.split(" ", 1)[1]
    raise RuntimeError(
        "Could not obtain a bearer token from the configured Databricks SDK auth. "
        "Configure an approved Databricks authentication method for this notebook."
    )


def space_id_from(value: str, label: str) -> str:
    """Accept a bare Genie Space ID or a URL that contains one (Genie UI link or MCP URL)."""
    value = value.strip()
    match = re.search(r"/(?:genie/rooms|mcp/genie)/([0-9A-Za-z]+)", value)
    if match:
        return match.group(1)
    if re.fullmatch(r"[0-9A-Za-z]+", value):
        return value
    raise ValueError(f"{label} = '{value}' is not a Genie Space ID or URL (https://<workspace>/genie/rooms/<id>).")


def validate_config():
    global GENIE_A_ID, GENIE_B_ID
    missing = [label for label, value in [("GENIE_A_ID", GENIE_A_ID), ("GENIE_B_ID", GENIE_B_ID)]
               if value.strip() in PLACEHOLDERS]
    if missing:
        raise ValueError("Please configure in section 2: " + ", ".join(missing))
    GENIE_A_ID = space_id_from(GENIE_A_ID, "GENIE_A_ID")
    GENIE_B_ID = space_id_from(GENIE_B_ID, "GENIE_B_ID")
    if GENIE_A_ID == GENIE_B_ID:
        raise ValueError("Genie Space A and B must be different Space IDs.")


def make_llm_client():
    return OpenAI(
        api_key=current_bearer_token(),
        base_url=f"{WORKSPACE_HOST}/serving-endpoints",
    )


def run_async(coro):
    """Run a coroutine from a notebook cell, whether or not an event loop is already running there."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


def show(df: pd.DataFrame):
    try:
        display(df)  # noqa: F821  (built in to Databricks and Jupyter)
    except NameError:
        print(df.to_string(index=False) if len(df) else "(no rows)")


validate_config()
print("Workspace:", WORKSPACE_HOST)
print("Signed in as:", w.current_user.me().user_name)
print("Genie A:", GENIE_A_ID)
print("Genie B:", GENIE_B_ID)
print("MCP Python SDK:", "2.x" if MCP_V2 else "1.x")

## 4. Define the two MCP client connections

This notebook connects to Databricks-managed Genie MCP endpoints. It does not start an MCP server or deploy an
application. One MCP session is opened per Genie question, and the question and its polls run in that session.

In [ ]:
def genie_mcp_url(space_id: str) -> str:
    return f"{WORKSPACE_HOST}/api/2.0/mcp/genie/{space_id}"


@asynccontextmanager
async def mcp_session(space_id: str):
    """An initialized MCP session with the Genie Space's managed MCP server."""
    url = genie_mcp_url(space_id)
    if MCP_V2:
        async with httpx2.AsyncClient(headers=current_auth_headers(), follow_redirects=True,
                                      timeout=httpx2.Timeout(MCP_TIMEOUT_SECONDS)) as http:
            async with Client(streamable_http_client(url, http_client=http)) as session:
                yield session
    else:
        async with streamablehttp_client(url, headers=current_auth_headers(), timeout=MCP_TIMEOUT_SECONDS,
                                         sse_read_timeout=MCP_TIMEOUT_SECONDS) as (read_stream, write_stream, _):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                yield session


def tool_info(tool):
    return {
        "name": tool.name,
        "description": tool.description or "",
        "inputSchema": getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None) or {},
    }


def tool_result(result):
    content = []
    for item in (getattr(result, "content", None) or []):
        content.append(item.text if hasattr(item, "text") else str(item))
    return {
        "isError": bool(getattr(result, "isError", getattr(result, "is_error", False))),
        "structuredContent": getattr(result, "structuredContent", getattr(result, "structured_content", None)),
        "content": content,
    }


async def mcp_list_tools(space_id: str):
    async with mcp_session(space_id) as session:
        result = await session.list_tools()
        return [tool_info(tool) for tool in result.tools]


async def mcp_call_tool(space_id: str, tool_name: str, arguments: dict, session=None):
    if session is None:
        async with mcp_session(space_id) as own_session:
            return tool_result(await own_session.call_tool(tool_name, arguments))
    return tool_result(await session.call_tool(tool_name, arguments))


def root_cause(err: BaseException) -> BaseException:
    """anyio wraps failures in exception groups; return the error that explains what happened."""
    while getattr(err, "exceptions", None):     # ExceptionGroup (Python 3.11+, or the exceptiongroup backport)
        err = err.exceptions[0]
    return err

## 5. Discover tool schemas from both Genie MCP endpoints

The orchestrator uses the actual discovered tool names and schemas. A Genie Space's MCP server offers a
**query tool** (`query_space_<id>`, which asks a question) and a **poll tool** (`poll_response_<id>`, which fetches
the answer once Genie has finished). They are found by name first, then by their arguments.

The space's title and description are read from the Genie API. Together with your optional settings, this is
what the router uses to decide.

In [ ]:
def likely_query_tool(tools):
    for tool in tools:                           # Databricks naming: query_space_<space id>
        if tool["name"].lower().startswith("query_space"):
            return tool
    for tool in tools:                           # otherwise: a tool whose name says ask/query, and is no poll tool
        name = tool["name"].lower()
        if any(word in name for word in ("query", "ask", "question")) and not any(
            word in name for word in ("poll", "cancel", "result", "get_query")
        ):
            return tool
    for tool in tools:                           # otherwise: a tool that takes a question
        props = (tool.get("inputSchema") or {}).get("properties") or {}
        if any(key in props for key in ("question", "query")) and "message_id" not in props:
            return tool
    return None


def likely_poll_tool(tools):
    for tool in tools:
        if tool["name"].lower().startswith("poll_response") or "poll" in tool["name"].lower():
            return tool
    return None


def infer_query_args(tool, question: str):
    schema = tool.get("inputSchema") or {}
    props = schema.get("properties") or {}
    required = schema.get("required") or []
    for key in ("question", "query", "content", "prompt", "input", "message"):
        if key in props:
            args = {key: question}
            missing = [name for name in required if name not in args]
            if missing:
                raise ValueError(
                    f"Tool {tool['name']} needs additional required arguments {missing}. "
                    f"Schema: {json.dumps(schema)}"
                )
            return args
    raise ValueError(f"Cannot infer question argument. Inspect schema: {json.dumps(schema)}")


def genie_space_info(space_id: str):
    """Title, description and sample questions of a Genie Space (sample questions need CAN EDIT; skipped if not)."""
    path = f"/api/2.0/genie/spaces/{space_id}"
    try:
        space = w.api_client.do("GET", path, query={"include_serialized_space": "true"})
    except Exception:
        try:
            space = w.api_client.do("GET", path)
        except Exception as err:                 # the MCP endpoint is what matters; carry on without the details
            print(f"Note: could not read the details of space {space_id}: {err}")
            return {"title": "", "description": "", "sample_questions": []}
    samples = []
    try:
        config = json.loads(space.get("serialized_space") or "{}").get("config") or {}
        for q in config.get("sample_questions") or []:
            text = q.get("question")
            text = " ".join(text) if isinstance(text, list) else text
            if text:
                samples.append(text)
    except (ValueError, AttributeError):
        pass
    return {"title": space.get("title") or "", "description": space.get("description") or "",
            "sample_questions": samples}


async def discover_both():
    tools_a, tools_b = await asyncio.gather(
        mcp_list_tools(GENIE_A_ID),
        mcp_list_tools(GENIE_B_ID),
    )
    return tools_a, tools_b


try:
    tools_a, tools_b = run_async(discover_both())
except Exception as err:
    raise RuntimeError(f"Could not list the tools of the Genie MCP endpoints: {root_cause(err)!r}") from None

SPACES = {}
for key, space_id, tools, name, description in [
    ("A", GENIE_A_ID, tools_a, GENIE_A_NAME, GENIE_A_DESCRIPTION),
    ("B", GENIE_B_ID, tools_b, GENIE_B_NAME, GENIE_B_DESCRIPTION),
]:
    info = genie_space_info(space_id)
    query_tool, poll_tool = likely_query_tool(tools), likely_poll_tool(tools)
    if query_tool is None:
        raise RuntimeError(f"No query tool found for space {key} among {[t['name'] for t in tools]}. "
                           "Inspect the schemas below and update likely_query_tool().")
    about = description.strip() or info["description"] or "(no description: set GENIE_%s_DESCRIPTION)" % key
    if info["sample_questions"]:
        about += " Example questions: " + " | ".join(info["sample_questions"][:5])
    SPACES[key] = {"id": space_id, "name": name.strip() or info["title"] or f"Genie Space {key}", "about": about,
                   "tools": tools, "query_tool": query_tool, "poll_tool": poll_tool,
                   "sample_questions": info["sample_questions"]}

show(pd.DataFrame([{"space": k, "name": s["name"], "query tool": s["query_tool"]["name"],
                    "poll tool": s["poll_tool"]["name"] if s["poll_tool"] else "(none)",
                    "what the router reads": s["about"][:200]} for k, s in SPACES.items()]))
print(json.dumps({"A": tools_a, "B": tools_b}, indent=2)[:4000])

## 6. Add the LLM router

The LLM is asked to return strict JSON specifying which Genie Space(s) to call and the question for each. The
notebook validates the route against an allow-list, so the model cannot invent tools or endpoints. If the reply
is not valid JSON, the router asks once more.

The routing rules are built from the spaces' names and descriptions (section 5). Adjust them to reflect your
actual domain ownership.

In [ ]:
def pick_orchestrator_endpoint():
    if ORCHESTRATOR_ENDPOINT.strip() not in PLACEHOLDERS:
        return ORCHESTRATOR_ENDPOINT.strip()
    preference = [r"databricks-claude-sonnet", r"databricks-claude-opus", r"databricks-claude", r"databricks-gpt-5",
                  r"databricks-meta-llama-3-3-70b", r"databricks-llama-4", r"databricks-gemini", r"databricks-"]
    chat = sorted((e.name for e in w.serving_endpoints.list() if "chat" in str(getattr(e, "task", "") or "")),
                  reverse=True)
    for pattern in preference:
        for name in chat:
            if re.match(pattern, name):
                return name
    if chat:
        return chat[0]
    raise RuntimeError("No chat endpoint found: set ORCHESTRATOR_ENDPOINT in section 2 (Serving > Endpoints).")


ORCHESTRATOR_ENDPOINT = pick_orchestrator_endpoint()
print("Orchestrator endpoint:", ORCHESTRATOR_ENDPOINT)


def router_system_prompt():
    a, b = SPACES["A"], SPACES["B"]
    return f'''
You are an enterprise data question router. Decide which existing Genie Space tools should answer the user's question.

Available spaces:
- Space A: "{a["name"]}" (space ID {a["id"]}). Data it holds: {a["about"]}
- Space B: "{b["name"]}" (space ID {b["id"]}). Data it holds: {b["about"]}

Rules:
- Choose A, B, BOTH or NONE.
- Choose BOTH when the question explicitly compares or combines domains, or requires evidence from both. Then give
  each space only the part of the question its data can answer.
- Choose NONE when neither space holds the data the question needs.
- Do not answer the business question yourself.
- Do not invent data, tools, or facts.
- Return ONLY valid JSON, with no other text, in this shape:
  {{
    "route": "A" | "B" | "BOTH" | "NONE",
    "question_a": "question to send to A or empty string",
    "question_b": "question to send to B or empty string",
    "reason": "brief routing rationale"
  }}
'''


def call_orchestrator_llm(messages, temperature=0):
    client = make_llm_client()
    try:
        response = client.chat.completions.create(
            model=ORCHESTRATOR_ENDPOINT,
            messages=messages,
            temperature=temperature,
        )
    except openai.BadRequestError as err:        # some reasoning models only accept the default temperature
        if "temperature" not in str(err).lower():
            raise
        response = client.chat.completions.create(model=ORCHESTRATOR_ENDPOINT, messages=messages)
    content = response.choices[0].message.content
    if isinstance(content, list):                # some endpoints return content blocks
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return content or ""


def parse_router_json(text, question=""):
    # Accepts markdown fences or a sentence around the JSON, while still requiring a valid JSON object.
    match = re.search(r"\{.*\}", (text or "").strip(), flags=re.DOTALL)
    if not match:
        raise ValueError(f"The router did not return JSON: {text[:300]!r}")
    obj = json.loads(match.group(0))
    obj["route"] = str(obj.get("route", "")).strip().upper()
    if obj["route"] not in {"A", "B", "BOTH", "NONE"}:
        raise ValueError(f"Invalid route returned by LLM: {obj.get('route')}")
    for key, needed in (("question_a", obj["route"] in {"A", "BOTH"}), ("question_b", obj["route"] in {"B", "BOTH"})):
        obj[key] = str(obj.get(key) or "").strip()
        if needed and not obj[key]:
            obj[key] = question                  # the router chose the space but left its question empty
        if not needed:
            obj[key] = ""
    return obj


def route_question(question):
    messages = [
        {"role": "system", "content": router_system_prompt()},
        {"role": "user", "content": question},
    ]
    reply = call_orchestrator_llm(messages)
    try:
        return parse_router_json(reply, question)
    except ValueError as err:                    # one retry, telling the model what was wrong
        messages += [{"role": "assistant", "content": reply or "(empty)"},
                     {"role": "user", "content": f"That was not valid: {err}. Return ONLY the JSON object."}]
        return parse_router_json(call_orchestrator_llm(messages), question)

## 7. Call the selected Genie tool(s)

This function executes only the route selected by the LLM, and only with the tool names discovered from each MCP
endpoint. The query tool starts a Genie message. While its status is not final, the poll tool is called with the
conversation and message IDs until Genie has written and run the SQL. The answer, the SQL and the rows are
returned as evidence for the synthesis step.

In [ ]:
TERMINAL = {"COMPLETED", "SUCCEEDED", "FAILED", "ERROR", "CANCELLED", "CANCELED", "QUERY_RESULT_EXPIRED",
            "INCOMPLETE"}


def payload_of(result):
    structured = result.get("structuredContent")
    if structured is not None:
        return structured
    text = "\n".join(result.get("content") or [])
    try:
        return json.loads(text)
    except ValueError:
        return text


def payload_text(result):
    payload = payload_of(result)
    return payload if isinstance(payload, str) else json.dumps(payload, ensure_ascii=False, default=str)


def _camel(name):
    head, *rest = name.split("_")
    return head + "".join(part.title() for part in rest)


def poll_args(poll_tool, ids):
    schema = poll_tool.get("inputSchema") or {}
    args = {}
    for name in schema.get("properties") or {}:
        for key in (name, _camel(name)):
            if ids.get(key) is not None:
                args[name] = ids[key]
                break
    missing = [name for name in schema.get("required") or [] if name not in args]
    if missing:
        raise RuntimeError(f"Cannot poll: the Genie reply has no {missing}: {json.dumps(ids, default=str)[:300]}")
    return args


def summarize_genie(payload, max_rows=50):
    """The parts of a Genie reply the LLM needs: status, text answer, SQL and the first rows."""
    if not isinstance(payload, dict):
        return {"status": "COMPLETED", "answer": str(payload), "queries": []}
    content = payload.get("content") or {}
    if isinstance(content, str):
        content = {"textAttachments": [content]}
    texts = [t if isinstance(t, str) else (t.get("content") or t.get("text") or json.dumps(t))
             for t in content.get("textAttachments") or []]
    queries = []
    for q in content.get("queryAttachments") or []:
        resp = q.get("statement_response") or q.get("statementResponse") or {}
        manifest = resp.get("manifest") or {}
        columns = [c.get("name") for c in (manifest.get("schema") or {}).get("columns") or []]
        rows = []
        for raw in (resp.get("result") or {}).get("data_array") or []:
            if isinstance(raw, dict) and "values" in raw:    # [{"string_value": ...}, ...]
                rows.append([next(iter(v.values()), None) if isinstance(v, dict) else v for v in raw["values"]])
            else:
                rows.append(list(raw) if isinstance(raw, (list, tuple)) else [raw])
        queries.append({"description": q.get("description"), "sql": q.get("query"), "columns": columns,
                        "rows": rows[:max_rows], "row_count": manifest.get("total_row_count", len(rows))})
    summary = {"status": str(payload.get("status") or "COMPLETED").upper(), "answer": " ".join(texts),
               "queries": queries}
    if payload.get("error"):
        summary["error"] = payload["error"]
    if not texts and not queries and not payload.get("error"):
        summary["answer"] = json.dumps(payload, default=str)[:4000]   # an unknown shape: pass it on as is
    conversation = payload.get("conversationId") or payload.get("conversation_id")
    if conversation:
        summary["conversation_id"] = conversation
    return summary


async def call_space(space_key, question):
    space = SPACES[space_key]
    tool = space["query_tool"]
    args = infer_query_args(tool, question)
    started, polls = time.time(), 0
    async with mcp_session(space["id"]) as session:
        result = await mcp_call_tool(space["id"], tool["name"], args, session=session)
        if result["isError"]:
            raise RuntimeError(f"MCP tool {tool['name']} returned an error: {payload_text(result)[:500]}")
        payload, ids = payload_of(result), {}
        while space["poll_tool"] and isinstance(payload, dict) and \
                str(payload.get("status") or "COMPLETED").upper() not in TERMINAL:
            if time.time() - started > GENIE_TIMEOUT_SECONDS:
                raise TimeoutError(f"Genie did not finish within {GENIE_TIMEOUT_SECONDS} s")
            ids.update({k: v for k, v in payload.items() if v is not None})
            await asyncio.sleep(POLL_EVERY_SECONDS)
            result = await mcp_call_tool(space["id"], space["poll_tool"]["name"],
                                         poll_args(space["poll_tool"], ids), session=session)
            if result["isError"]:
                raise RuntimeError(f"MCP tool {space['poll_tool']['name']} returned an error: "
                                   f"{payload_text(result)[:500]}")
            payload, polls = payload_of(result), polls + 1
    if isinstance(payload, dict) and ids:
        payload = {**{k: v for k, v in ids.items() if k != "status"}, **payload}
    return {
        "space": space["name"],
        "space_id": space["id"],
        "tool": tool["name"],
        "arguments": args,
        "seconds": round(time.time() - started, 1),
        "polls": polls,
        "genie": summarize_genie(payload),
    }


async def execute_route(route):
    tasks = []
    labels = []
    if route["route"] in {"A", "BOTH"}:
        labels.append("A")
        tasks.append(call_space("A", route["question_a"]))
    if route["route"] in {"B", "BOTH"}:
        labels.append("B")
        tasks.append(call_space("B", route["question_b"]))

    results = await asyncio.gather(*tasks, return_exceptions=True)
    return {
        label: ({"space": SPACES[label]["name"], "error": repr(root_cause(result))}
                if isinstance(result, Exception) else result)
        for label, result in zip(labels, results)
    }

## 8. Synthesize a final answer with the LLM

The same orchestrator model receives the user's original question and the results returned by the selected Genie
Space(s). It must distinguish evidence from each space and disclose when a tool failed or did not return enough
information.

In [ ]:
SYNTHESIS_SYSTEM_PROMPT = '''
You are an enterprise analytics assistant. Answer using only the supplied Genie tool results.
- Attribute findings to the relevant Genie Space when useful.
- If results disagree, explicitly explain the discrepancy rather than silently choosing one.
- Do not invent metrics, SQL results, citations, or business explanations not supported by the tool results.
- If a tool failed or returned insufficient information, state the limitation.
- If no Genie Space was called, say that the question is outside the data of both spaces.
- Keep the answer clear and concise.
'''


def orchestrate(question: str):
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    route = route_question(question)

    # Safety/traceability: only execute routes A/B/BOTH with the discovered tools of the two configured spaces.
    tool_results = run_async(execute_route(route)) if route["route"] != "NONE" else {}

    synthesis_input = {
        "user_question": question,
        "routing_decision": route,
        "genie_tool_results": tool_results,
    }
    final_answer = call_orchestrator_llm([
        {"role": "system", "content": SYNTHESIS_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(synthesis_input, ensure_ascii=False, default=str)},
    ])

    return {
        "question": question,
        "routing": route,
        "tool_results": tool_results,
        "final_answer": final_answer,
    }


def print_result(result):
    route = result["routing"]
    print("QUESTION:", result["question"])
    print(f"ROUTING:  {route['route']}  ({route.get('reason', '')})")
    for label, r in result["tool_results"].items():
        if "error" in r:
            print(f"  Space {label} ({r['space']}): ERROR {r['error']}")
            continue
        g = r["genie"]
        print(f"  Space {label} ({r['space']}) via MCP {r['tool']}: {g['status']} in {r['seconds']} s, "
              f"{r['polls']} poll(s)")
        print(f"    asked: {r['arguments']}")
        for q in g["queries"]:
            print("    SQL:", (q["sql"] or "").strip().replace("\n", "\n         "))
            if q["columns"]:
                show(pd.DataFrame(q["rows"], columns=q["columns"]).head(20))
        if g.get("conversation_id"):
            print(f"    open in Genie: {WORKSPACE_HOST}/genie/rooms/{r['space_id']}/chats/{g['conversation_id']}")
    print("FINAL ANSWER:\n" + result["final_answer"].strip())

## 9. Run the demo

The demo asks one question for Space A, one for Space B, one that needs BOTH, and one that NONE of them can
answer, then shows how each was routed. Without `DEMO_QUESTIONS`, the questions are the spaces' own sample
questions, or are written by the LLM from the space descriptions.

In [ ]:
def write_question(prompt):
    text = call_orchestrator_llm([{"role": "user", "content": prompt}]).strip().strip('"').splitlines()
    return text[0].strip().strip('"') if text else ""


def describe(key):
    return f'"{SPACES[key]["name"]}": {SPACES[key]["about"]}'


demo_questions = [(item, "?") if isinstance(item, str) else (item[0], item[1].upper()) for item in DEMO_QUESTIONS]
if not demo_questions:
    for key in ("A", "B"):
        question = (SPACES[key]["sample_questions"] or [None])[0] or write_question(
            f"A Databricks Genie space holds this data: {describe(key)}\n"
            "Write ONE short, concrete business question it can answer. Reply with the question only.")
        demo_questions.append((question, key))
    both = write_question(
        f"Two Databricks Genie spaces:\n- {describe('A')}\n- {describe('B')}\n"
        "Write ONE short business question that needs data from BOTH spaces. Reply with the question only.")
    demo_questions.append((both, "BOTH"))
    demo_questions.append(("What is the capital of France?", "NONE"))

DEMO_RESULTS = []
for question, expected in demo_questions:
    print("\n" + "=" * 100)
    result = orchestrate(question)
    print_result(result)
    DEMO_RESULTS.append({"question": question, "expected": expected, "routed": result["routing"]["route"],
                         "as expected": "?" if expected == "?" else
                         ("yes" if expected == result["routing"]["route"] else "NO")})

print("\n" + "=" * 100 + "\nRouting summary")
show(pd.DataFrame(DEMO_RESULTS))

### Ask your own question

In [ ]:
MY_QUESTION = ""   # e.g. "Compare sales trends with the stock-outs in the same stores."

if MY_QUESTION.strip():
    print_result(orchestrate(MY_QUESTION))
else:
    print("Set MY_QUESTION above and run this cell again.")

## 10. Important implementation notes

- **No new deployment:** this notebook only calls existing managed Genie MCP endpoints and an existing
  model-serving endpoint. Section 1 installs Python packages into this notebook's session only. The only traces
  left are the conversations in each space's Genie history and the LLM's usage.
- **LLM endpoint:** the orchestration model is not created by this notebook. Set `ORCHESTRATOR_ENDPOINT` to an
  existing compatible Databricks Model Serving endpoint, or leave it empty to use a Foundation Model API endpoint.
  If your organization uses a different approved LLM provider, replace `make_llm_client()` and
  `call_orchestrator_llm()`.
- **Genie MCP schemas:** the query tool starts a Genie message and the poll tool returns the answer when Genie has
  finished; `call_space()` follows that flow. If your workspace exposes other tool names or arguments, inspect
  section 5's discovered schemas and adjust `likely_query_tool()`, `likely_poll_tool()` or `poll_args()`.
- **Routing quality:** the router decides on the spaces' names and descriptions. If routing is off, write a
  clearer `GENIE_A_DESCRIPTION` / `GENIE_B_DESCRIPTION`.
- **Governance:** the notebook identity must be authorized to access both Genie Spaces and their underlying data.
  Genie runs every query as you. The LLM sees the tool outputs passed to the synthesis step (up to 50 rows per
  query), so follow your organization's data handling policy.
- **Evaluation:** test routing accuracy, answer grounding, disagreements between spaces, latency, and model/tool
  cost before using this as a production pattern.

## Official documentation
- [Databricks managed MCP servers](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp)
- [Databricks Genie Agent MCP server](https://docs.databricks.com/aws/en/agents/mcp-tools/genie-agent)
- [Databricks Model Serving](https://docs.databricks.com/aws/en/machine-learning/model-serving/)
- [MCP Python SDK client transports](https://py.sdk.modelcontextprotocol.io/client/transports/)